# SC-PaCMAP data preparation

This notebook creates the two processed real datasets used by the paper benchmarks.

## Dataset 1: Critical Minerals in Ores (CMiO)

Geoscience Australia geochemistry is filtered to one UTM zone and prepared as a deposit-group benchmark with 35 chemical features.

## Dataset 2: North Batten drill-core geochemistry

The supplied 2015, 2018, and 2019 collar, survey, geology, assay, and lithology-code files are combined into a three-dimensional lithology benchmark with 46 shared `ME-MS61` measurements.

The synthetic benchmark generates its data inside its own notebook and requires no preparation input. Labels produced here are excluded from embedding and clustering and are used only for evaluation.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repository_root(start: Path) -> Path:
    ancestors = (start.resolve(), *start.resolve().parents)
    candidates = (*ancestors, *(path / "sc-pacmap-reproducibility" for path in ancestors))
    for candidate in candidates:
        if (candidate / "data_preparation").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the sc-pacmap-reproducibility repository.")


ROOT = find_repository_root(Path.cwd())
RAW_DIR = ROOT / "data" / "raw"
OUTPUT_DIR = ROOT / "data" / "processed"
NORTH_BATTEN_RAW_DIR = RAW_DIR / "north_batten"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_CMIO_CLASS_SIZE = 20
MIN_NORTH_BATTEN_CLASS_SIZE = 20
MIN_NORTH_BATTEN_FEATURE_COVERAGE = 0.80

CMIO_SOURCE_PAGE = "https://ecat.ga.gov.au/geonetwork/srv/api/records/3c876198-5f78-4f6a-836e-da7f22c3333e"
NORTH_BATTEN_SOURCE_PAGE = "https://geoscience.nt.gov.au/gemis/ntgsjspui/handle/1/94531"
#https://geoscience.nt.gov.au/gemis/ntgsjspui/handle/1/91634
#https://geoscience.nt.gov.au/gemis/ntgsjspui/handle/1/94065
#https://geoscience.nt.gov.au/gemis/ntgsjspui/handle/1/94531
# Updated file mappings to match actual NT Geological Survey naming conventions
NORTH_BATTEN_FILES = {
    2015: {
        "collars": "GR321_2015_GA_03_Appendix2.txt",
        "surveys": "GR321_2015_GA_04_Appendix3.txt",
        "geology": "GR321_2015_GA_05_Appendix4.txt",
        "assays": "GR321_2015_GA_06_Appendix5.txt",
        "codes": "GR321_2015_GA_07_Appendix6.txt",
    },
    2018: {
        "collars": "GR321_2018_GA_02_NBA_NTSL4_COLL2018A.txt",
        "surveys": "GR321_2018_GA_03_NBA_NTDS4_SURV2018A.txt",
        "geology": "GR321_2018_GA_04_NBA_NTDL4_GEO2018A.txt",
        "assays": "GR321_2018_GA_06_NBA_NTDG4_ASS2018A_1.txt",
        "codes": "GR321_2018_GA_10_NBA_LithCodes.csv",
    },
    2019: {
        "collars": "GR504_2019_A_04_DrillCollars_revised.txt",
        "surveys": "GR504_2019_A_08_Survey_revised.txt",
        "geology": "GR504_2019_A_07_Geol_revised.txt",
        "assays": "GR504_2019_A_06_Assay_revised.txt",
        "codes": "GR504_2019_A_12_LithCodes_revised.CSV",
    },
}

print("Repository:", ROOT)


## 1. Check the supplied raw files

The notebook uses the local source CSV files in `raw`. Missing files raise a clear error so the preparation remains reproducible and never silently substitutes another dataset.

In [ ]:
CMIO_FILE = RAW_DIR / "cmio" / "CriticalMineralDepositsGeochemistry.csv"
required_files = [CMIO_FILE]
for year, files in NORTH_BATTEN_FILES.items():
    required_files.extend(
        NORTH_BATTEN_RAW_DIR / str(year) / filename
        for filename in files.values()
    )

missing_files = [str(path.relative_to(ROOT)) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing raw source files. Follow data/raw/README.md and add: "
        f"{missing_files}"
    )

print("CMiO file found:", CMIO_FILE.name)
print("North Batten files found:", sum(len(files) for files in NORTH_BATTEN_FILES.values()))

# Part A - Critical Minerals in Ores

The cleaning is intentionally simple:

1. Keep one UTM zone (`54S`) so X and Y share one coordinate system.
2. Use the existing `DEPOSIT_GROUP` field as the discrete label.
3. Keep 35 well-populated chemical variables and retain complete rows across that panel.
4. Interpret each negative assay as a below-detection-limit (BDL) code, following the source geochemistry convention.
5. Replace each coded value $x < 0$ with $|x| / 2$. The encoded magnitude is used because companion detection-limit metadata are incomplete for some legacy records; this is left-censor substitution, not missing-value imputation.
6. Calculate `z` as negative interval midpoint depth.

In [ ]:
CMIO_FEATURES = [
    "Y_PPM", "V_PPM", "LA_PPM", "U_PPM", "SR_PPM", "SN_PPM",
    "RB_PPM", "TH_PPM", "CS_PPM", "CE_PPM", "NB_PPM", "GA_PPM",
    "W_PPM", "HF_PPM", "TA_PPM", "ZR_PPM", "CR_PPM", "PR_PPM",
    "ND_PPM", "ER_PPM", "EU_PPM", "SM_PPM", "YB_PPM", "TB_PPM",
    "HO_PPM", "GD_PPM", "DY_PPM", "LU_PPM", "BA_PPM",
    "SIO2_WT_PERCENT", "TIO2_WT_PERCENT", "NA2O_WT_PERCENT",
    "K2O_WT_PERCENT", "AL2O3_WT_PERCENT", "TM_PPM",
]

cmio_raw = pd.read_csv(CMIO_FILE, low_memory=False)
print("Raw CMiO shape:", cmio_raw.shape)

In [ ]:
# Keep one compatible projected coordinate system.
cmio = cmio_raw.loc[
    cmio_raw["SAMPLE_UTM_ZONE"].astype(str).str.strip().eq("54S")
].copy()

required_numeric = [
    "SAMPLE_EASTING", "SAMPLE_NORTHING", "TOP_DEPTH_M", "BASE_DEPTH_M",
    *CMIO_FEATURES,
]

# Convert bad text values to NaN, then keep complete rows.
for column in required_numeric:
    cmio[column] = pd.to_numeric(cmio[column], errors="coerce")

cmio["LABEL_deposit_group"] = cmio["DEPOSIT_GROUP"].astype("string").str.strip()
cmio = cmio.dropna(subset=["LABEL_deposit_group", *required_numeric]).copy()
cmio = cmio.loc[cmio["LABEL_deposit_group"].ne("")].copy()

# GA negative assay codes denote below-detection-limit (BDL) observations.
# Substitute half the encoded limit (-v -> v / 2); this handles left censoring,
# not missing data, and avoids incomplete legacy companion detection-limit fields.
cmio[CMIO_FEATURES] = cmio[CMIO_FEATURES].mask(
    cmio[CMIO_FEATURES] < 0,
    cmio[CMIO_FEATURES].abs() / 2,
)

# Keep labels that have enough examples for evaluation.
class_counts = cmio["LABEL_deposit_group"].value_counts()
keep_labels = class_counts[class_counts >= MIN_CMIO_CLASS_SIZE].index
cmio = cmio.loc[cmio["LABEL_deposit_group"].isin(keep_labels)].copy()

# Build common spatial and identifier columns.
cmio["sample_id"] = cmio["SAMPLE_UID"].astype(str)
cmio["group_id"] = cmio["DEPOSIT_NAME"].astype(str)
cmio["x"] = cmio["SAMPLE_EASTING"]
cmio["y"] = cmio["SAMPLE_NORTHING"]
cmio["z"] = -(cmio["TOP_DEPTH_M"] + cmio["BASE_DEPTH_M"]) / 2

label_order = sorted(cmio["LABEL_deposit_group"].unique())
label_map = {label: number for number, label in enumerate(label_order)}
cmio["label_id"] = cmio["LABEL_deposit_group"].map(label_map).astype(int)

cmio_ready = cmio[
    [
        "sample_id", "group_id", "x", "y", "z",
        "LABEL_deposit_group", "label_id", *CMIO_FEATURES,
    ]
].reset_index(drop=True)

print("Clean CMiO shape:", cmio_ready.shape)
print("Number of features:", len(CMIO_FEATURES))
print("Number of classes:", cmio_ready["LABEL_deposit_group"].nunique())
print(cmio_ready["LABEL_deposit_group"].value_counts())

In [ ]:
# Save only the cleaned geochemistry dataset.
cmio_output = OUTPUT_DIR / "cmio_sc_pacmap_ready.csv"
cmio_ready.to_csv(cmio_output, index=False)

print("Saved:", cmio_output)

In [ ]:
def add_interval_label(samples, intervals, label_column):
    """Assign a label when one geology interval covers at least half the sample."""
    output = []

    for drill_id, sample_group in samples.groupby("DRILL_ID", sort=False):
        interval_group = intervals.loc[intervals["DRILL_ID"].eq(drill_id)].copy()
        interval_group = interval_group.dropna(subset=["FROM", "TO", label_column])
        if interval_group.empty:
            continue

        overlap = np.maximum(
            0,
            np.minimum(sample_group["to_m"].to_numpy()[:, None], interval_group["TO"].to_numpy())
            - np.maximum(sample_group["from_m"].to_numpy()[:, None], interval_group["FROM"].to_numpy()),
        )
        positions = overlap.argmax(axis=1)
        sample_lengths = sample_group["to_m"].to_numpy() - sample_group["from_m"].to_numpy()
        valid = (sample_lengths > 0) & (overlap.max(axis=1) >= 0.5 * sample_lengths)

        matched = sample_group.iloc[np.flatnonzero(valid)].copy()
        matched["label_lithology"] = interval_group.iloc[positions[valid]][label_column].to_numpy()
        output.append(matched)

    if not output:
        return samples.iloc[0:0].assign(label_lithology=pd.Series(dtype="string"))
    return pd.concat(output, ignore_index=True)

## Shared interval and desurvey helpers

The preparation uses interval-overlap labels and minimum-curvature desurveying. When the first survey station is below zero depth, its measured direction is extended back to the collar. Samples beyond the final station are flagged and excluded.

In [ ]:
def prepare_survey(hole_surveys):
    """Return clean measured-depth, dip, and grid-azimuth survey stations."""
    data = hole_surveys[["Depth", "Dip", "Azimuth"]].copy()
    data.columns = ["DEPTH", "DIP", "azimuth_deg"]
    data = data.apply(pd.to_numeric, errors="coerce").dropna()
    data = data.loc[
        data["DEPTH"].ge(0)
        & data["DIP"].between(-90, 90)
        & data["azimuth_deg"].between(0, 360, inclusive="left")
    ]
    data = data.sort_values("DEPTH").drop_duplicates("DEPTH", keep="last")

    if data.empty:
        return data
    if data.iloc[0]["DEPTH"] > 0:
        first_station = data.iloc[[0]].copy()
        first_station["DEPTH"] = 0.0
        data = pd.concat([first_station, data], ignore_index=True)

    return data


def minimum_curvature_xyz(sample_depths, survey_rows, collar_x, collar_y, collar_z):
    """Calculate XYZ at requested measured depths using minimum curvature."""
    sample_depths = np.asarray(sample_depths, dtype=float)
    station_depths = survey_rows["DEPTH"].to_numpy(float)
    station_dips = survey_rows["DIP"].to_numpy(float)
    station_azimuths = survey_rows["azimuth_deg"].to_numpy(float)

    # Source dip is negative down; inclination is measured from vertical down.
    station_inclinations = np.radians(90.0 + station_dips)
    station_azimuths = np.unwrap(np.radians(station_azimuths))

    nodes = np.unique(np.round(np.concatenate(([0.0], station_depths, sample_depths)), 6))
    inclinations = np.interp(nodes, station_depths, station_inclinations)
    azimuths = np.interp(nodes, station_depths, station_azimuths)

    east = np.zeros(len(nodes))
    north = np.zeros(len(nodes))
    down = np.zeros(len(nodes))

    for index in range(1, len(nodes)):
        length = nodes[index] - nodes[index - 1]
        inc_1, inc_2 = inclinations[index - 1], inclinations[index]
        azi_1, azi_2 = azimuths[index - 1], azimuths[index]

        cosine_dogleg = (
            np.cos(inc_1) * np.cos(inc_2)
            + np.sin(inc_1) * np.sin(inc_2) * np.cos(azi_2 - azi_1)
        )
        dogleg = np.arccos(np.clip(cosine_dogleg, -1.0, 1.0))
        ratio_factor = 1.0 if dogleg < 1e-12 else 2.0 * np.tan(dogleg / 2.0) / dogleg

        north[index] = north[index - 1] + (
            length / 2.0
            * (np.sin(inc_1) * np.cos(azi_1) + np.sin(inc_2) * np.cos(azi_2))
            * ratio_factor
        )
        east[index] = east[index - 1] + (
            length / 2.0
            * (np.sin(inc_1) * np.sin(azi_1) + np.sin(inc_2) * np.sin(azi_2))
            * ratio_factor
        )
        down[index] = down[index - 1] + (
            length / 2.0
            * (np.cos(inc_1) + np.cos(inc_2))
            * ratio_factor
        )

    sample_positions = np.searchsorted(nodes, np.round(sample_depths, 6))
    x = collar_x + east[sample_positions]
    y = collar_y + north[sample_positions]
    z = collar_z - down[sample_positions]
    return x, y, z

# Part C - North Batten drill-core geochemistry

This section combines the supplied 2015, 2018, and 2019 campaigns with transparent rules:

1. Read the Northern Territory tabular templates from their `H1000` data headers.
2. Keep direct `ME-MS61` measurements shared by all three campaigns.
3. Keep features present in at least 80% of chemistry-bearing rows in every campaign, then use complete rows only.
4. Replace a reported `<v` result with `v / 2` and a reported `>v` result with `v`.
5. Assign the primary lithology when one geology interval covers at least half of the assay interval.
6. Convert lithology codes to readable labels and keep final classes with at least 20 intervals.
7. Calculate MGA94 Zone 53 `x` and `y` by minimum curvature. `z` is vertical displacement below the collar because later collar elevations are missing or zero placeholders.
8. Exclude samples beyond the final survey station. No assay imputation or survey extrapolation is used.

In [ ]:
def read_nt_template(path):
    """Read one NT exploration template without losing duplicate headers."""
    header_rows = {}
    data_rows = []

    with path.open(encoding="utf-8-sig") as source:
        for line_number, line in enumerate(source, start=1):
            parts = line.rstrip("\r\n").split("\t")
            row_type = parts[0].strip()
            if row_type in {"H1000", "H1001", "H1002", "H1003"}:
                header_rows[row_type] = parts[1:]
            elif row_type == "D":
                data_rows.append((line_number, parts[1:]))

    if "H1000" not in header_rows:
        raise ValueError(f"No H1000 data header found in {path.name}")

    source_names = header_rows["H1000"]
    width = len(source_names)
    column_keys = [f"column_{position:03d}" for position in range(width)]

    clean_rows = []
    for line_number, values in data_rows:
        if len(values) > width and any(value.strip() for value in values[width:]):
            raise ValueError(f"Too many fields in {path.name} at line {line_number}")
        clean_rows.append((values + [""] * width)[:width])

    data = pd.DataFrame(clean_rows, columns=column_keys)
    schema = pd.DataFrame({
        "column_key": column_keys,
        "source_name": source_names,
        "unit": (header_rows.get("H1001", []) + [""] * width)[:width],
        "method": (header_rows.get("H1002", []) + [""] * width)[:width],
        "detection_limit": (header_rows.get("H1003", []) + [""] * width)[:width],
    })
    return data, schema


def simple_name(value):
    """Make source header spelling differences easy to compare."""
    return "".join(character.lower() for character in str(value) if character.isalnum())


def nt_column_key(schema, aliases):
    wanted = {simple_name(alias) for alias in aliases}
    matches = schema.loc[
        schema["source_name"].map(simple_name).isin(wanted), "column_key"
    ].tolist()
    if len(matches) != 1:
        raise ValueError(f"Expected one of {aliases}, found {len(matches)} matches")
    return matches[0]


def concentration_unit(value):
    unit = str(value).strip().lower()
    if unit in {"ppm", "partspermillion"}:
        return "PPM"
    if unit in {"%", "percent", "pct"}:
        return "PCT"
    return None


def assay_number(value, detection_limit):
    """Convert source assay text while preserving documented censoring."""
    text = str(value).strip()
    if not text or text.upper() in {"NULL", "NA", "N/A", "NAN", "-"}:
        return np.nan

    multiplier = 1.0
    if text.startswith("<"):
        multiplier = 0.5
        text = text[1:].strip()
    elif text.startswith(">"):
        text = text[1:].strip()

    try:
        number = float(text) * multiplier
    except ValueError:
        return np.nan

    limit = pd.to_numeric(detection_limit, errors="coerce")
    if number < 0 and pd.notna(limit) and np.isclose(abs(number), limit):
        return abs(number) / 2.0
    return number


def prepare_north_batten_assays(year, path):
    """Return valid assay intervals and comparable ME-MS61 features."""
    raw, schema = read_nt_template(path)
    hole_key = nt_column_key(schema, ["Hole_id"])
    sample_key = nt_column_key(schema, ["Sample_id", "Sample ID"])
    from_key = nt_column_key(schema, ["From", "Depth From"])
    to_key = nt_column_key(schema, ["To", "Depth To"])

    output = pd.DataFrame({
        "source_year": year,
        "source_order": np.arange(len(raw)),
        "DRILL_ID": raw[hole_key].astype("string").str.strip(),
        "source_sample_id": raw[sample_key].astype("string").str.strip(),
        "from_m": pd.to_numeric(raw[from_key], errors="coerce"),
        "to_m": pd.to_numeric(raw[to_key], errors="coerce"),
    })

    feature_sources = []
    for row in schema.itertuples(index=False):
        unit = concentration_unit(row.unit)
        if str(row.method).strip().upper() != "ME-MS61" or unit is None:
            continue
        analyte = simple_name(row.source_name).upper()
        if not analyte:
            continue
        feature_sources.append(
            (row.column_key, f"{analyte}_ME_MS61_{unit}", row.detection_limit)
        )

    feature_names = [feature for _, feature, _ in feature_sources]
    if len(feature_names) != len(set(feature_names)):
        raise ValueError(f"Duplicate normalized ME-MS61 features in {path.name}")

    for column_key, feature, detection_limit in feature_sources:
        output[feature] = raw[column_key].map(
            lambda value, limit=detection_limit: assay_number(value, limit)
        )

    valid_interval = (
        output["DRILL_ID"].notna()
        & output["DRILL_ID"].ne("")
        & output["from_m"].notna()
        & output["to_m"].notna()
        & output["from_m"].lt(output["to_m"])
    )
    output = output.loc[valid_interval].copy()
    output = output.loc[output[feature_names].notna().any(axis=1)].copy()

    print(
        f"{year}: {len(raw)} source assay rows, {len(output)} valid rows "
        f"with ME-MS61 chemistry, {len(feature_names)} candidate features"
    )
    return output, feature_names

In [ ]:
north_batten_assays_by_year = {}
north_batten_candidates_by_year = {}

for year, files in NORTH_BATTEN_FILES.items():
    assay_path = NORTH_BATTEN_RAW_DIR / str(year) / files["assays"]
    assays, candidate_features = prepare_north_batten_assays(year, assay_path)
    north_batten_assays_by_year[year] = assays
    north_batten_candidates_by_year[year] = set(candidate_features)

shared_features = sorted(set.intersection(*north_batten_candidates_by_year.values()))
if not shared_features:
    raise ValueError("No method-and-unit-qualified ME-MS61 features are shared by all campaigns")

coverage_by_year = pd.DataFrame({
    year: assays[shared_features].notna().mean()
    for year, assays in north_batten_assays_by_year.items()
})
NORTH_BATTEN_FEATURES = coverage_by_year.index[
    coverage_by_year.ge(MIN_NORTH_BATTEN_FEATURE_COVERAGE).all(axis=1)
].tolist()

if len(NORTH_BATTEN_FEATURES) < 2:
    raise ValueError("The North Batten coverage rule retained fewer than two features")

print("\nShared ME-MS61 features:", len(shared_features))
print("Coverage by campaign for retained features:")
print(coverage_by_year.loc[NORTH_BATTEN_FEATURES].round(3).to_string())
print("Retained features:", NORTH_BATTEN_FEATURES)

clean_assay_parts = []
for year, assays in north_batten_assays_by_year.items():
    assays = assays.copy()
    assays["observed_features"] = assays[NORTH_BATTEN_FEATURES].notna().sum(axis=1)
    interval_keys = ["source_year", "DRILL_ID", "from_m", "to_m"]
    duplicate_rows = int(assays.duplicated(interval_keys, keep=False).sum())
    assays = (
        assays.sort_values([*interval_keys, "observed_features", "source_order"])
        .drop_duplicates(interval_keys, keep="last")
        .drop(columns="observed_features")
    )
    before_complete_case = len(assays)
    assays = assays.dropna(subset=NORTH_BATTEN_FEATURES).copy()
    print(
        f"{year}: {duplicate_rows} duplicate interval rows considered; "
        f"{len(assays)} of {before_complete_case} unique rows are complete"
    )
    clean_assay_parts.append(assays)

north_batten = pd.concat(clean_assay_parts, ignore_index=True)
if set(north_batten["source_year"].unique()) != set(NORTH_BATTEN_FILES):
    raise ValueError("Complete-case filtering removed an entire North Batten campaign")
if (north_batten[NORTH_BATTEN_FEATURES] < 0).any().any():
    raise ValueError("Negative values found in retained North Batten chemistry")

north_batten["depth_m"] = (north_batten["from_m"] + north_batten["to_m"]) / 2
fallback_id = (
    north_batten["DRILL_ID"].astype(str)
    + "_"
    + north_batten["from_m"].map(lambda value: f"{value:.4f}")
    + "_"
    + north_batten["to_m"].map(lambda value: f"{value:.4f}")
)
source_id = north_batten["source_sample_id"].fillna("").str.strip()
source_id = source_id.mask(source_id.eq(""), fallback_id)
north_batten["sample_id"] = north_batten["source_year"].astype(str) + "_" + source_id
north_batten["group_id"] = north_batten["DRILL_ID"].astype(str)

if north_batten["sample_id"].duplicated().any():
    raise ValueError("Duplicate North Batten sample identifiers remain")

print("\nComplete comparable assay rows:", len(north_batten))
print(north_batten["source_year"].value_counts().sort_index())

In [ ]:
def prepare_north_batten_geology(year, path):
    """Return valid primary-lithology intervals for one campaign."""
    raw, schema = read_nt_template(path)
    hole_key = nt_column_key(schema, ["Hole_id"])
    from_key = nt_column_key(schema, ["DEPTH_FROM", "Depth From"])
    to_key = nt_column_key(schema, ["DEPTH_TO", "Depth To"])
    lithology_key = nt_column_key(schema, ["Lithology"])

    geology = pd.DataFrame({
        "source_year": year,
        "DRILL_ID": raw[hole_key].astype("string").str.strip(),
        "FROM": pd.to_numeric(raw[from_key], errors="coerce"),
        "TO": pd.to_numeric(raw[to_key], errors="coerce"),
        "lithology_code": raw[lithology_key].astype("string").str.strip().str.upper(),
    })
    valid = (
        geology["DRILL_ID"].notna()
        & geology["FROM"].notna()
        & geology["TO"].notna()
        & geology["FROM"].lt(geology["TO"])
        & geology["lithology_code"].notna()
        & geology["lithology_code"].ne("")
    )
    return geology.loc[valid].copy()


def add_north_batten_lithology(samples, geology):
    """Assign the largest geology overlap when it covers half the assay."""
    output = []
    group_columns = ["source_year", "DRILL_ID"]

    for group_key, sample_group in samples.groupby(group_columns, sort=False):
        year, drill_id = group_key
        interval_group = geology.loc[
            geology["source_year"].eq(year) & geology["DRILL_ID"].eq(drill_id)
        ]
        if interval_group.empty:
            continue

        overlap = np.maximum(
            0,
            np.minimum(sample_group["to_m"].to_numpy()[:, None], interval_group["TO"].to_numpy())
            - np.maximum(sample_group["from_m"].to_numpy()[:, None], interval_group["FROM"].to_numpy()),
        )
        positions = overlap.argmax(axis=1)
        sample_lengths = sample_group["to_m"].to_numpy() - sample_group["from_m"].to_numpy()
        valid = overlap.max(axis=1) >= 0.5 * sample_lengths

        matched = sample_group.iloc[np.flatnonzero(valid)].copy()
        matched["lithology_code"] = interval_group.iloc[positions[valid]][
            "lithology_code"
        ].to_numpy()
        output.append(matched)

    if not output:
        return samples.iloc[0:0].assign(lithology_code=pd.Series(dtype="string"))
    return pd.concat(output, ignore_index=True)


geology_parts = []
code_parts = []
for year, files in NORTH_BATTEN_FILES.items():
    year_folder = NORTH_BATTEN_RAW_DIR / str(year)
    geology_parts.append(
        prepare_north_batten_geology(year, year_folder / files["geology"])
    )

    separator = "\t" if year_folder.joinpath(files["codes"]).suffix.lower() == ".txt" else ","
    codes = pd.read_csv(year_folder / files["codes"], sep=separator, dtype="string")
    codes.columns = [column.strip() for column in codes.columns]
    codes = codes.loc[
        codes["FieldName"].str.strip().str.upper().eq("LITHOLOGY"),
        ["Code", "Description"],
    ].copy()
    codes["source_year"] = year
    codes["lithology_code"] = codes["Code"].str.strip().str.upper()
    codes["description"] = codes["Description"].str.strip()
    code_parts.append(codes[["source_year", "lithology_code", "description"]])

north_batten_geology = pd.concat(geology_parts, ignore_index=True)
all_lithology_codes = pd.concat(code_parts, ignore_index=True)
description_counts = all_lithology_codes.groupby("lithology_code")["description"].nunique()
conflicting_codes = description_counts[description_counts > 1].index.tolist()
if conflicting_codes:
    print("Lithology descriptions updated between campaigns:", conflicting_codes)

# Prefer the most recent supplied description when a codebook was updated.
lithology_lookup = (
    all_lithology_codes.sort_values("source_year")
    .drop_duplicates("lithology_code", keep="last")
    .set_index("lithology_code")["description"]
)

rows_before_geology = len(north_batten)
north_batten = add_north_batten_lithology(north_batten, north_batten_geology)
non_rock_codes = {"CLS", "CVT", "NCP", "NL", "NR"}
non_rock_rows = int(north_batten["lithology_code"].isin(non_rock_codes).sum())
north_batten["lithology_description"] = north_batten["lithology_code"].map(lithology_lookup)
unmapped_rows = int(north_batten["lithology_description"].isna().sum())
north_batten = north_batten.loc[
    ~north_batten["lithology_code"].isin(non_rock_codes)
    & north_batten["lithology_description"].notna()
].copy()
north_batten["label_lithology"] = (
    north_batten["lithology_code"] + " - " + north_batten["lithology_description"]
)

print("Assay rows before geology matching:", rows_before_geology)
print("Rows with at least 50% primary-lithology coverage:", len(north_batten) + non_rock_rows + unmapped_rows)
print("Excluded non-rock rows:", non_rock_rows)
print("Excluded unmapped lithology rows:", unmapped_rows)
print("Rows with readable rock labels:", len(north_batten))
print(north_batten["label_lithology"].value_counts())

In [ ]:
def optional_nt_column_key(schema, aliases):
    wanted = {simple_name(alias) for alias in aliases}
    matches = schema.loc[
        schema["source_name"].map(simple_name).isin(wanted), "column_key"
    ].tolist()
    return matches[0] if len(matches) == 1 else None


def nt_metadata_value(path, record_name):
    with path.open(encoding="utf-8-sig") as source:
        for line in source:
            parts = line.rstrip("\r\n").split("\t")
            if simple_name(parts[0]) == simple_name(record_name):
                return parts[2].strip() if len(parts) > 2 else ""
    return ""


def prepare_north_batten_collars(year, path):
    """Return unique MGA94 Zone 53 collar XY positions."""
    datum = nt_metadata_value(path, "H0501").upper()
    zone = nt_metadata_value(path, "H0531")
    if datum != "GDA94" or zone != "53":
        raise ValueError(f"Unexpected coordinate system in {path.name}: {datum}, zone {zone}")

    raw, schema = read_nt_template(path)
    hole_key = nt_column_key(schema, ["Hole_id"])
    x_key = nt_column_key(schema, ["MGA_E", "Easting_MGA"])
    y_key = nt_column_key(schema, ["MGA_N", "Northing_MGA"])
    collars = pd.DataFrame({
        "source_year": year,
        "DRILL_ID": raw[hole_key].astype("string").str.strip(),
        "collar_x": pd.to_numeric(raw[x_key], errors="coerce"),
        "collar_y": pd.to_numeric(raw[y_key], errors="coerce"),
    }).dropna(subset=["DRILL_ID", "collar_x", "collar_y"])

    if collars.duplicated(["source_year", "DRILL_ID"]).any():
        raise ValueError(f"Duplicate valid collars in {path.name}")
    return collars


def prepare_north_batten_surveys(year, path):
    """Return measured survey stations and their source azimuth columns."""
    raw, schema = read_nt_template(path)
    hole_key = nt_column_key(schema, ["Hole_id"])
    depth_key = nt_column_key(schema, ["Surveyed_Depth", "Surveyed Depth"])
    dip_key = nt_column_key(schema, ["Dip"])
    grid_key = optional_nt_column_key(schema, ["Azimuth_grid"])
    magnetic_key = optional_nt_column_key(schema, ["Azimith_mag", "Azimuth_MAG"])

    surveys = pd.DataFrame({
        "source_year": year,
        "DRILL_ID": raw[hole_key].astype("string").str.strip(),
        "Depth": pd.to_numeric(raw[depth_key], errors="coerce"),
        "Dip": pd.to_numeric(raw[dip_key], errors="coerce"),
        "azimuth_grid_source": (
            pd.to_numeric(raw[grid_key], errors="coerce") if grid_key else np.nan
        ),
        "azimuth_magnetic": (
            pd.to_numeric(raw[magnetic_key], errors="coerce") if magnetic_key else np.nan
        ),
    })
    return surveys


collar_parts = []
survey_parts = []
for year, files in NORTH_BATTEN_FILES.items():
    year_folder = NORTH_BATTEN_RAW_DIR / str(year)
    collar_parts.append(
        prepare_north_batten_collars(year, year_folder / files["collars"])
    )
    survey_parts.append(
        prepare_north_batten_surveys(year, year_folder / files["surveys"])
    )

north_batten_collars = pd.concat(collar_parts, ignore_index=True)
north_batten_surveys = pd.concat(survey_parts, ignore_index=True)

reference_rows = north_batten_surveys.loc[
    north_batten_surveys["source_year"].eq(2015)
    & north_batten_surveys["azimuth_grid_source"].notna()
    & north_batten_surveys["azimuth_magnetic"].notna()
    & (
        north_batten_surveys["azimuth_grid_source"].ne(0)
        | north_batten_surveys["azimuth_magnetic"].ne(0)
    )
].copy()
reference_rows["grid_minus_magnetic"] = (
    reference_rows["azimuth_grid_source"]
    - reference_rows["azimuth_magnetic"]
    + 180.0
) % 360.0 - 180.0
magnetic_to_grid = float(reference_rows["grid_minus_magnetic"].median())
correction_spread = float(
    (reference_rows["grid_minus_magnetic"] - magnetic_to_grid).abs().max()
)
if len(reference_rows) < 10 or correction_spread > 1.0:
    raise ValueError("The 2015 magnetic-to-grid correction is not sufficiently stable")

north_batten_surveys["Azimuth"] = north_batten_surveys["azimuth_grid_source"]
north_batten_surveys["azimuth_reference"] = "SOURCE_GRID"
later_campaign = north_batten_surveys["source_year"].ne(2015)
north_batten_surveys.loc[later_campaign, "Azimuth"] = (
    north_batten_surveys.loc[later_campaign, "azimuth_magnetic"] + magnetic_to_grid
) % 360.0
north_batten_surveys.loc[later_campaign, "azimuth_reference"] = (
    "MAGNETIC_PLUS_2015_OFFSET"
)

print(
    "Magnetic-to-grid correction:",
    f"{magnetic_to_grid:.2f} degrees from {len(reference_rows)} paired stations;",
    f"maximum deviation {correction_spread:.2f} degrees",
)

collar_lookup = north_batten_collars.set_index(["source_year", "DRILL_ID"])
coordinate_parts = []
missing_collar_rows = 0
missing_survey_rows = 0
beyond_survey_rows = 0

for group_key, samples in north_batten.groupby(["source_year", "DRILL_ID"], sort=False):
    year, drill_id = group_key
    if group_key not in collar_lookup.index:
        missing_collar_rows += len(samples)
        continue

    hole_surveys = north_batten_surveys.loc[
        north_batten_surveys["source_year"].eq(year)
        & north_batten_surveys["DRILL_ID"].eq(drill_id)
    ]
    survey_rows = prepare_survey(hole_surveys)
    if survey_rows.empty:
        missing_survey_rows += len(samples)
        continue

    within_survey = samples["depth_m"].le(survey_rows["DEPTH"].max())
    beyond_survey_rows += int((~within_survey).sum())
    samples = samples.loc[within_survey].copy()
    if samples.empty:
        continue

    collar = collar_lookup.loc[group_key]
    samples["x"], samples["y"], samples["z"] = minimum_curvature_xyz(
        samples["depth_m"].to_numpy(),
        survey_rows,
        float(collar["collar_x"]),
        float(collar["collar_y"]),
        0.0,
    )
    samples["azimuth_reference"] = hole_surveys["azimuth_reference"].iloc[0]
    samples["survey_extrapolated"] = False
    coordinate_parts.append(samples)

if not coordinate_parts:
    raise ValueError("No North Batten intervals have valid collar and survey data")
north_batten = pd.concat(coordinate_parts, ignore_index=True)

final_counts = north_batten["label_lithology"].value_counts()
final_labels = final_counts[final_counts >= MIN_NORTH_BATTEN_CLASS_SIZE].index
north_batten_ready = north_batten.loc[
    north_batten["label_lithology"].isin(final_labels)
].copy()

label_order = sorted(north_batten_ready["label_lithology"].unique())
label_map = {label: number for number, label in enumerate(label_order)}
north_batten_ready["label_id"] = (
    north_batten_ready["label_lithology"].map(label_map).astype(int)
)

output_columns = [
    "sample_id", "group_id", "source_year", "depth_m", "x", "y", "z",
    "lithology_code", "label_lithology", "label_id", "azimuth_reference",
    "survey_extrapolated", *NORTH_BATTEN_FEATURES,
]
north_batten_ready = north_batten_ready[output_columns].reset_index(drop=True)

if set(north_batten_ready["source_year"].unique()) != set(NORTH_BATTEN_FILES):
    raise ValueError("Coordinate or class filtering removed an entire North Batten campaign")
if (north_batten_ready["z"] > 1e-9).any():
    raise ValueError("Collar-relative North Batten z must be nonpositive")

print("Rows excluded for missing collars:", missing_collar_rows)
print("Rows excluded for missing surveys:", missing_survey_rows)
print("Rows excluded beyond final survey stations:", beyond_survey_rows)
print("Final North Batten shape:", north_batten_ready.shape)
print("Final assay features:", len(NORTH_BATTEN_FEATURES))
print("Final lithology classes:", north_batten_ready["label_lithology"].nunique())
print("Final drillholes:", north_batten_ready["group_id"].nunique())
print("Rows by source year:")
print(north_batten_ready["source_year"].value_counts().sort_index())
print("Final class counts:")
print(north_batten_ready["label_lithology"].value_counts())

In [ ]:
north_batten_output = OUTPUT_DIR / "north_batten_sc_pacmap_ready.csv"
north_batten_ready.to_csv(north_batten_output, index=False)

print("Saved:", north_batten_output)

# Important paper notes

- **CMiO:** `x` and `y` are source UTM Zone 54S coordinates. `z` is negative interval midpoint depth, not absolute elevation.
- **North Batten:** `x` and `y` are MGA94 Zone 53 coordinates. `z` is minimum-curvature vertical displacement relative to the collar (`z = 0`), not AHD elevation.
- North Batten combines the supplied 2015, 2018, and 2019 campaigns. Later magnetic surveys are converted to grid bearings with the median offset measured from paired 2015 source bearings.
- North Batten features are shared direct `ME-MS61` measurements with at least 80% coverage in every campaign. Censored results use half the detection limit, and rows are complete across all retained features.
- Feature values remain unscaled. Standardisation is performed in each benchmark.

# Final checks

These checks verify that both outputs are immediately usable for ML: no missing or non-finite values, unique sample identifiers, no constant features, multiple classes, all three North Batten campaigns represented, no negative retained North Batten concentrations, and no survey extrapolation.

In [ ]:
def validate_dataset(name, data, features, label_column):
    required = [
        "sample_id", "group_id", "x", "y", "z",
        label_column, "label_id", *features,
    ]
    numeric = data[["x", "y", "z", *features]].to_numpy(dtype=float)
    result = {
        "dataset": name,
        "rows": len(data),
        "features": len(features),
        "classes": data[label_column].nunique(),
        "groups": data["group_id"].nunique(),
        "missing_required_values": int(data[required].isna().sum().sum()),
        "nonfinite_numeric_values": int((~np.isfinite(numeric)).sum()),
        "duplicate_sample_ids": int(data["sample_id"].duplicated().sum()),
        "constant_features": int(data[features].nunique(dropna=False).le(1).sum()),
    }

    assert result["rows"] > 0
    assert result["features"] > 0
    assert result["classes"] > 1
    assert result["missing_required_values"] == 0
    assert result["nonfinite_numeric_values"] == 0
    assert result["duplicate_sample_ids"] == 0
    assert result["constant_features"] == 0
    return result


summary = pd.DataFrame([
    validate_dataset(
        "CMiO geochemistry", cmio_ready, CMIO_FEATURES, "LABEL_deposit_group"
    ),
    validate_dataset(
        "North Batten assays",
        north_batten_ready,
        NORTH_BATTEN_FEATURES,
        "label_lithology",
    ),
])

assert north_batten_ready["label_lithology"].value_counts().min() >= MIN_NORTH_BATTEN_CLASS_SIZE
assert set(north_batten_ready["source_year"].unique()) == set(NORTH_BATTEN_FILES)
assert not north_batten_ready["survey_extrapolated"].any()
assert not (north_batten_ready[NORTH_BATTEN_FEATURES] < 0).any().any()
assert north_batten_ready["x"].between(100_000, 900_000).all()
assert north_batten_ready["y"].between(0, 10_000_000).all()
assert north_batten_ready["z"].le(1e-9).all()

straight_survey = pd.DataFrame({
    "DEPTH": [0.0, 100.0],
    "DIP": [-90.0, -90.0],
    "azimuth_deg": [0.0, 0.0],
})
straight_x, straight_y, straight_z = minimum_curvature_xyz(
    np.array([100.0]), straight_survey, 0.0, 0.0, 0.0
)
assert np.allclose([straight_x[0], straight_y[0], straight_z[0]], [0.0, 0.0, -100.0])

summary

In [ ]:
# Quick visual check of both final class distributions.
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

cmio_ready["LABEL_deposit_group"].value_counts().sort_values().plot.barh(ax=axes[0])
axes[0].set_title("CMiO deposit-group counts")
axes[0].set_xlabel("Samples")

north_batten_ready["label_lithology"].value_counts().sort_values().plot.barh(ax=axes[1])
axes[1].set_title("North Batten lithology counts")
axes[1].set_xlabel("Intervals")

plt.tight_layout()
plt.show()

# Load arrays for SC-PaCMAP

Use the saved feature lists so identifiers, coordinates, and labels are never accidentally passed into the feature matrix. Fit imputation or scaling only inside each training fold.

In [ ]:
cmio_data = pd.read_csv(OUTPUT_DIR / "cmio_sc_pacmap_ready.csv")
north_batten_data = pd.read_csv(OUTPUT_DIR / "north_batten_sc_pacmap_ready.csv")

X_cmio = cmio_data[CMIO_FEATURES].to_numpy(dtype=float)
xyz_cmio = cmio_data[["x", "y", "z"]].to_numpy(dtype=float)
y_cmio = cmio_data["label_id"].to_numpy(dtype=int)

X_north_batten = north_batten_data[NORTH_BATTEN_FEATURES].to_numpy(dtype=float)
xyz_north_batten = north_batten_data[["x", "y", "z"]].to_numpy(dtype=float)
y_north_batten = north_batten_data["label_id"].to_numpy(dtype=int)

for name, features, coordinates, labels in [
    ("CMiO", X_cmio, xyz_cmio, y_cmio),
    ("North Batten", X_north_batten, xyz_north_batten, y_north_batten),
]:
    assert len(features) == len(coordinates) == len(labels)
    assert np.isfinite(features).all()
    assert np.isfinite(coordinates).all()
    print(name + ":", features.shape, coordinates.shape, labels.shape)